Author: **Dongyuan Gao**

Course: HSLU Computer Vision — Lecture 3 Project

Based on the style of the lecturer's notebooks by *Safouane El Ghazouali* (TOELT LLC / HSLU).

# -----  -----  -----  -----  -----  -----  -----  -----

# 🚗 Fine-Tuning YOLOv10 for Self-Driving Object Detection

In this notebook we fine-tune a pre-trained **YOLOv10n** on the **Udacity Self-Driving Car dataset** (≈15,000 dashcam images, 11 classes: car, truck, pedestrian, biker, traffic light, traffic sign, …).

The goal: build a **real-time detector** that finds vehicles, pedestrians and traffic signals in dashcam footage — the core perception step of any self-driving stack.

### Why YOLO?
- **Real-time**: a single forward pass gives all boxes + labels.
- **Detection, not classification**: tells us *what* AND *where*.
- **Easy fine-tuning** via the Ultralytics library.

### What You'll Learn
- Downloading a pre-labelled detection dataset from Roboflow.
- Fine-tuning YOLOv10n on custom classes.
- Reading training curves and validation metrics (mAP).
- Running inference on images and videos.
- Plugging the fine-tuned weights into a **live webcam** demo on your Mac.

# 🧭 Running on DGX via VS Code Remote

Project directory on DGX: `/home/dongyuan/Desktop/computer_vision`

Typical flow:
- Connect to the DGX with VS Code Remote - SSH.
- Open this notebook **on the remote machine** (so paths refer to DGX storage).
- Use a conda env or venv with PyTorch + CUDA already installed.
- Keep datasets on DGX local storage (faster than network mounts).

# 🧰 Environment Setup (DGX)

Install Ultralytics (YOLO), Roboflow (dataset download), and OpenCV.

On a DGX, you typically already have a CUDA-enabled PyTorch in your conda env.
If you do not, create or activate your environment before running the install below.

In [ ]:
!pip install -q ultralytics roboflow opencv-python

### Import Libraries & Check GPU

On the DGX you should see `cuda` and at least one visible GPU.
If it prints `cpu`, your environment is missing CUDA-enabled PyTorch or no GPU is visible.

In [ ]:
from ultralytics import YOLO
from roboflow import Roboflow
import torch
import os, glob, yaml
import cv2
import matplotlib.pyplot as plt
from PIL import Image
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

# Quick GPU visibility check on DGX
!nvidia-smi -L

# Explanation
# - device: tells YOLO where to run (GPU is ~30x faster than CPU).
# - Ultralytics auto-uses this device unless we override it.

# 📂 Dataset on the DGX (Roboflow or Local Path)

You can either download with Roboflow **on the DGX** or point to a dataset that is already on DGX storage.

**Option A (Roboflow download on DGX):**
1. Go to https://public.roboflow.com/object-detection/self-driving-car
2. Click **Download Dataset** → pick **YOLOv8** format (compatible with v10).
3. Roboflow shows you a **personalized snippet** with your API key — paste it in the next cell.

**Option B (Dataset already on DGX):**
- Set the `DATASET_DIR` path below to the folder that contains `data.yaml`, `train/`, `valid/`, `test/`.

In [ ]:
# Set this to False if the dataset is already on DGX storage
USE_ROBOFLOW = True

# If USE_ROBOFLOW is False, set the local dataset folder on DGX
DATASET_DIR = "/home/dongyuan/Desktop/computer_vision/self-driving-car"

if USE_ROBOFLOW:
    # ---- PASTE YOUR ROBOFLOW SNIPPET HERE ----
    rf = Roboflow(api_key="YOUR_API_KEY_HERE")
    project = rf.workspace("roboflow-gw7yv").project("self-driving-car")
    dataset = project.version(3).download("yolov8")
    dataset_location = dataset.location
else:
    dataset_location = DATASET_DIR

data_yaml = os.path.join(dataset_location, "data.yaml")
print(f"Dataset location: {dataset_location}")
print(f"data.yaml: {data_yaml}")

# Explanation
# - dataset_location: absolute path to the dataset folder on DGX
# - data.yaml lists class names and the train/valid/test paths YOLO needs

### Inspect the Dataset

Before training, always look at a few images to confirm labels make sense.
YOLO labels are **normalised** (values between 0 and 1) in the format:

````
class  x_center  y_center  width  height
````

We convert them back to pixel coordinates to draw the boxes.

In [ ]:
images_dir = f"{dataset_location}/train/images"
labels_dir = f"{dataset_location}/train/labels"

# Read class names from data.yaml
with open(data_yaml) as f:
    data_cfg = yaml.safe_load(f)
class_names = data_cfg["names"]
print(f"Classes ({len(class_names)}):", class_names)


def plot_image_with_boxes(image_path, label_path):
    img = cv2.imread(image_path)
    h, w, _ = img.shape

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f.readlines():
                cls, x, y, bw, bh = map(float, line.strip().split())
                cls = int(cls)
                # YOLO normalized -> pixel coords
                x1 = int((x - bw / 2) * w)
                y1 = int((y - bh / 2) * h)
                x2 = int((x + bw / 2) * w)
                y2 = int((y + bh / 2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    img,
                    class_names[cls],
                    (x1, max(y1 - 5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2,
                )

    plt.figure(figsize=(8, 5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()


# Show 3 sample images
sample_files = sorted(os.listdir(images_dir))[:3]
for file in sample_files:
    img_path = os.path.join(images_dir, file)
    lbl_path = os.path.join(labels_dir, file.rsplit(".", 1)[0] + ".txt")
    plot_image_with_boxes(img_path, lbl_path)

# Explanation
# - This is a sanity check and not part of training.
# - If labels do not match the objects visually, check data.yaml and paths.

# 📦 Loading a Pre-trained YOLOv10n Model

We start from **`yolov10n.pt`** (nano = smallest, fastest) pretrained on **COCO** (80 everyday classes).

Fine-tuning means: **reuse COCO's learned features**, then adjust the output head for our 11 self-driving classes. That is why we don't need millions of images — the model already knows what a "car" looks like from COCO.

Model size options:
- `yolov10n.pt` → Nano (≈3M params) — fastest, fine for demos ✅
- `yolov10s.pt` → Small (≈7M params) — better accuracy
- `yolov10m.pt` → Medium (≈15M params) — slower but strongest for a class project

In [ ]:
model = YOLO('yolov10n.pt')
print('YOLOv10n loaded with COCO pretrained weights!')

# 🏋️ Fine-Tuning the Model

Key training arguments:

| Arg | Meaning | Our value |
|---|---|---|
| `data` | Path to data.yaml | DGX path |
| `epochs` | Number of passes over the dataset | 30 |
| `imgsz` | Input image size (square) | 512 |
| `batch` | Images per training step | 16 |
| `patience` | Early stop if no improvement for N epochs | 10 |
| `device` | `cuda` = GPU, `cpu` = CPU | auto |

⏱️ **Expected time on a DGX GPU: 15-40 min (varies by GPU).**
Ultralytics prints a progress bar per epoch so you can see the ETA.

Results are auto-saved to `runs/detect/selfdriving_v1/` including:
- `weights/best.pt` -> best-performing weights
- `results.png` -> loss + mAP curves
- `confusion_matrix.png`

In [ ]:
results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=512,
    batch=16,
    patience=10,
    name="selfdriving_v1",
    device=device,
 )

# Save the run directory for later cells (paths are auto-incremented)
save_dir = str(model.trainer.save_dir)
print(f"\nRun directory: {save_dir}")

# Explanation
# - Fine-tuning = predict -> loss -> backward -> optimizer.step (same loop as the ViT notebook).
# - Ultralytics hides the loop inside .train(), but it is the same idea.
# - After training, `model` automatically holds the best weights.

### Look at the Training Curves

`results.png` shows loss (should decrease) and mAP (should increase) over epochs.
If mAP flatlined early, more epochs won't help much.

In [ ]:
results_png = os.path.join(save_dir, 'results.png')
if os.path.exists(results_png):
    plt.figure(figsize=(14, 8))
    plt.imshow(Image.open(results_png))
    plt.axis('off')
    plt.show()
else:
    print('results.png not found — training may still be in progress.')

# 📊 Part 1 — Formal Evaluation (the numbers for your report)

**Evaluation** = running the model on a held-out split and computing objective metrics.
We use `model.val()` on the **validation** split (Ultralytics uses the `val:` path from `data.yaml`).

**Key metrics:**
- **Precision** — of the boxes we predicted, how many were correct?
- **Recall** — of all real objects, how many did we find?
- **mAP@0.5** — mean Average Precision at IoU threshold 0.5 (main benchmark number).
- **mAP@0.5:0.95** — stricter; averaged across IoU thresholds 0.5 → 0.95.

A solid YOLOv10n on this dataset typically reaches **mAP@0.5 ≈ 0.55–0.70**.

In [ ]:
val_results = model.val()

print(f"\nPrecision     : {val_results.box.mp:.3f}")
print(f"Recall        : {val_results.box.mr:.3f}")
print(f"mAP@0.5       : {val_results.box.map50:.3f}")
print(f"mAP@0.5:0.95  : {val_results.box.map:.3f}")

# Confusion matrix (which classes get confused with which?)
cm_path = os.path.join(save_dir, "confusion_matrix.png")
if os.path.exists(cm_path):
    plt.figure(figsize=(10, 10))
    plt.imshow(Image.open(cm_path))
    plt.axis("off")
    plt.title("Confusion Matrix")
    plt.show()

# Explanation
# - val_results.box.* holds aggregated metrics over all classes.
# - For per-class metrics, check val_results.box.maps (one value per class).

# 🔍 Part 2 — Qualitative Demo on a Single Test Image

Numbers are nice, but it helps to **see** the model work. We pick a random image from the test split and plot the predicted boxes.

`conf=0.4` filters out low-confidence boxes to keep the output clean.

In [ ]:
test_dir = f"{dataset_location}/test/images"
test_img = os.path.join(test_dir, sorted(os.listdir(test_dir))[0])

results_test = model(test_img, conf=0.4)

annotated = results_test[0].plot()
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title("Fine-tuned YOLOv10 - Dashcam Detections")
plt.axis("off")
plt.show()

### Exploring the `results` Object

Ultralytics wraps predictions in a `Results` object. The most useful attributes:

| Attribute | What it contains |
|---|---|
| `.boxes.xyxy` | Box corners in pixels `[x1, y1, x2, y2]` |
| `.boxes.conf` | Confidence score (0–1) per box |
| `.boxes.cls` | Class index per box |
| `.names` | Dict mapping class index → class name |

In [ ]:
result = results_test[0]

boxes = result.boxes.xyxy.cpu().numpy()
confs = result.boxes.conf.cpu().numpy()
clses = result.boxes.cls.cpu().numpy().astype(int)
names = result.names

print(f'Found {len(boxes)} objects:\n')
for box, conf, cls in zip(boxes, confs, clses):
    print(f'  {names[cls]:15s}  conf={conf:.2f}  box={box.astype(int).tolist()}')

# 🎥 Part 3 — Video Demo (DGX Path Input)

Place a dashcam clip on the DGX (scp it from your Mac if needed).
The code below processes every frame and **saves an annotated output video** on the DGX.

In [ ]:
# Set a DGX path to a local video file
video_path = "/home/dongyuan/Desktop/computer_vision/videos/dashcam.mp4"
assert os.path.exists(video_path), "Video path not found on DGX"

# Run detection on every frame - saves to runs/detect/predict*/
model(video_path, save=True, conf=0.4, device=device)

# Find the annotated output and print its path
out_candidates = sorted(
    glob.glob("runs/detect/predict*/*.avi") + glob.glob("runs/detect/predict*/*.mp4"),
    key=os.path.getmtime,
 )
if out_candidates:
    out_video = out_candidates[-1]
    print(f"Output file: {out_video}")
else:
    print("No output video found - check runs/detect/ manually.")

# Explanation
# - Ultralytics writes .avi by default; recent versions may use .mp4.
# - Use scp to copy the output video back to your Mac if needed.

# 💾 Save the Fine-Tuned Weights (DGX)

Store `best.pt` in a durable DGX path (e.g., your home or project directory),
then copy it to your Mac with `scp` if you want to run the webcam demo locally.

In [ ]:
best_pt = os.path.join(save_dir, "weights", "best.pt")
target_dir = "/home/dongyuan/Desktop/computer_vision/yolo_selfdriving"
os.makedirs(target_dir, exist_ok=True)

!cp "{best_pt}" "{target_dir}/best.pt"
!cp "{save_dir}/results.png" "{target_dir}/results.png"
print("Saved best.pt and results.png to:", target_dir)

# To copy to your Mac, use scp from a terminal, e.g.:
# scp dongyuan@<dgx-host>:/home/dongyuan/Desktop/computer_vision/yolo_selfdriving/best.pt ./

# 📷 Part 4 — Live Webcam Demo on Your Mac

The DGX cannot access your Mac webcam. After training, copy `best.pt` from the DGX to your Mac,
then point your local `YOLO_Live.py` at those weights.

### Steps

1. Copy `best.pt` from DGX to your Mac with `scp`.
2. Place it next to `YOLO_Live.py`, e.g. in `Lecture 3/best.pt`.
3. Open `YOLO_Live.py` and change **one line**:

````python
# Before (uses the generic COCO-pretrained model):
model = YOLO('yolov10n.pt')

# After (uses YOUR fine-tuned self-driving model):
model = YOLO('best.pt')
````

4. Make sure you have `ultralytics` installed locally:

````bash
pip install ultralytics opencv-python
````

5. Run the script:

````bash
python3 YOLO_Live.py
````

6. Press **q** to quit the webcam window.

### What to Expect

Your webcam now detects the **11 self-driving classes** you fine-tuned on (car, truck, pedestrian, ...). If you point it at a phone/monitor playing a dashcam clip, or print out a street photo, the model should draw colored boxes in real time at roughly **20-40 FPS** on an M-series Mac.

# 🧠 Interpreting Results — What to Report

When presenting this project, cover:

1. **Dataset**: what it contains, how many images, how many classes.
2. **Model choice**: why YOLOv10n (speed vs accuracy trade-off).
3. **Training curves**: show `results.png` — did loss decrease? did mAP plateau?
4. **Metrics**: precision, recall, mAP@0.5. Note which classes were weakest (check confusion matrix).
5. **Qualitative results**: the annotated dashcam video and/or live webcam demo — the "wow" moment.
6. **Limitations**: small objects (far-away cars), night scenes, weather, class imbalance.
7. **Next steps**: more epochs, bigger model (`yolov10s`), stronger augmentation, add tracking.

# 💡 Student Tasks

1. **Train longer** — bump `epochs` to 50 with `patience=15`. Does mAP improve?
2. **Bigger model** — try `yolov10s.pt` or `yolo11n.pt` and compare training time vs mAP gain.
3. **Different dataset** — try **Tsinghua-Tencent TT100K** (traffic signs only) and compare.
4. **Add tracking** — replace `model(video_path, save=True)` with `model.track(video_path, tracker="bytetrack.yaml", save=True)` so each vehicle keeps a persistent ID across frames.
5. **Measure FPS** — time the video inference and compute frames/second. Is it real-time (>30 FPS)?
6. **Per-class analysis** — print `val_results.box.maps` and discuss which classes are hardest.